# 노트북 11. LangGraph 상태 그래프와 ReAct 에이전트 설계

> Phase 3 — 실전 기법: 챗봇을 똑똑하게 만드는 기술

노트북 9에서 도구 하나를 호출하는 법을 배웠습니다.
이제 여러 노드를 연결하고, 조건에 따라 분기하는 '에이전트 그래프'를 설계할 차례입니다.

**학습 목표**
- ReAct(Reasoning + Acting) 패턴의 원리를 이해한다
- LangGraph의 StateGraph, 노드, 엣지, 상태를 활용할 수 있다
- llm_node와 tool_node의 조건부 분기를 구현할 수 있다
- Checkpointer를 통해 대화 상태를 영속화할 수 있다
- 커스텀 노드를 추가하여 그래프를 확장할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | ReAct 패턴 + StateGraph 설계 + Checkpointer | 읽고 실행 |
| Part 2 — 실습 | 그래프 구성 + 조건부 분기 + ReAct 완성 | 직접 작성 |
| Part 3 — 챌린지 | 커스텀 노드 확장 에이전트 설계 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai langgraph

import os
import json
from google import genai
from google.genai import types

# API 키 설정 (Colab 환경 기준)
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

MODEL = "gemini-2.5-flash"

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
)

print("환경 설정 완료")

---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 ReAct 패턴이란?

**ReAct** = **Re**asoning + **Act**ing

모델이 다음을 **반복**합니다:
1. **Reasoning**: 현재 상황을 분석하고, 무엇을 해야 하는지 추론
2. **Acting**: 도구를 호출하여 행동
3. **Observation**: 도구의 결과를 관찰
4. 결과가 충분하면 최종 답변, 아니면 1로 돌아감

```
사용자 질문
    ↓
LLM: "날씨를 먼저 확인하자" (Reasoning)
    ↓
도구: get_weather("서울") → {temp: 15} (Acting + Observation)
    ↓
LLM: "기온에 맞는 옷을 추천하자" (Reasoning)
    ↓
도구: recommend_outfit(15) → "가디건" (Acting + Observation)
    ↓
LLM: "서울은 15도이므로 가디건을 추천합니다" (최종 답변)
```

### 노트북 9의 Tool Calling과의 차이

| 구분 | 노트북 9 (Tool Calling) | 노트북 11 (ReAct) |
|------|----------------------|-------------------|
| 도구 호출 | 1회 또는 병렬 1회 | 필요할 때까지 반복 |
| 분기 로직 | 없음 (단순 루프) | 조건부 엣지로 분기 |
| 상태 관리 | messages 리스트 수동 관리 | StateGraph 자동 관리 |
| 확장성 | 단일 루프 | 노드 추가로 유연하게 확장 |
| 적합한 경우 | 간단한 도구 호출 | 복잡한 다단계 작업 |

## 1.2 LangGraph StateGraph 기초

LangGraph의 핵심 구성 요소 3가지:

| 구성 요소 | 역할 | 예시 |
|-----------|------|------|
| **State** | 그래프 전체에서 공유하는 데이터 | `{"messages": [...]}` |
| **Node** | 상태를 받아 처리하고 반환하는 함수 | `llm_node`, `tool_node` |
| **Edge** | 노드 간 연결 (무조건/조건부) | `llm → tools` 또는 `llm → END` |

```python
from langgraph.graph import StateGraph, MessagesState, START, END

graph = StateGraph(MessagesState)    # State 정의
graph.add_node("llm", llm_func)      # Node 추가
graph.add_edge(START, "llm")         # Edge 연결
```

### MessagesState

`MessagesState`는 LangGraph가 제공하는 기본 상태 클래스입니다.
`messages` 필드 하나를 가지며, 메시지 리스트를 자동으로 누적합니다.

```python
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

`add_messages`는 **reducer** 함수입니다.
노드가 반환한 메시지를 기존 리스트에 **추가**합니다 (덮어쓰지 않음).

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

# 가장 간단한 그래프: 노드 1개
def echo_node(state: MessagesState):
    """마지막 메시지를 그대로 반환합니다."""
    last_msg = state["messages"][-1]
    return {"messages": [AIMessage(content=f"에코: {last_msg.content}")]}

# 그래프 구성
builder = StateGraph(MessagesState)
builder.add_node("echo", echo_node)
builder.add_edge(START, "echo")
builder.add_edge("echo", END)

graph = builder.compile()

# 실행
result = graph.invoke({"messages": [HumanMessage(content="안녕하세요")]})

for msg in result["messages"]:
    print(f"[{type(msg).__name__}] {msg.content}")

In [ ]:
# add_messages reducer의 동작 확인# 노드가 반환한 메시지가 기존 리스트에 "추가"되는 과정from langgraph.graph.message import add_messagesexisting = [HumanMessage(content="안녕")]new = [AIMessage(content="반갑습니다")]# add_messages는 기존 리스트에 새 메시지를 추가merged = add_messages(existing, new)print(f"기존: {[m.content for m in existing]}")print(f"추가: {[m.content for m in new]}")print(f"결과: {[m.content for m in merged]}")print(f"길이: {len(existing)} + {len(new)} = {len(merged)}")

> 노드 함수는 `state`를 받아서 **변경할 부분만** 반환합니다.
> `{"messages": [new_msg]}`를 반환하면, `add_messages` reducer가 기존 리스트에 추가합니다.

## 1.3 커스텀 State 정의

`MessagesState`는 `messages` 필드만 가집니다.
추가 필드가 필요하면 `TypedDict`로 커스텀 상태를 정의합니다.

> **메시지 덮어쓰기**: 같은 `id`를 가진 메시지를 반환하면, add_messages는 추가 대신 **교체**합니다.> 이를 활용하면 이전 메시지를 수정하거나, 요약으로 대체할 수 있습니다.>> ```python> # 기존 메시지의 id를 지정하여 교체> return {"messages": [AIMessage(content="수정됨", id=old_msg.id)]}> ```

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

# 커스텀 상태: messages + 추가 필드
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    turn_count: int       # 대화 턴 수
    user_name: str        # 사용자 이름

# 커스텀 상태를 사용하는 노드
def greet_node(state: ChatState):
    name = state.get("user_name", "사용자")
    turn = state.get("turn_count", 0) + 1
    return {
        "messages": [AIMessage(content=f"{name}님, 안녕하세요! (턴 {turn})")],
        "turn_count": turn,
    }

builder = StateGraph(ChatState)
builder.add_node("greet", greet_node)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)
graph = builder.compile()

result = graph.invoke({
    "messages": [HumanMessage(content="안녕")],
    "turn_count": 0,
    "user_name": "민수",
})

print(f"messages: {[m.content for m in result['messages']]}")
print(f"turn_count: {result['turn_count']}")

> `messages` 필드는 반드시 `Annotated[list[AnyMessage], add_messages]`로 정의해야
> LangGraph가 메시지를 올바르게 누적합니다.
> 다른 필드(turn_count, user_name)는 일반 타입으로 정의하며, 반환 시 덮어씁니다.

## 1.4 조건부 엣지

그래프의 핵심은 **조건에 따라 다른 노드로 분기**하는 것입니다.
`add_conditional_edges()`로 조건부 분기를 설정합니다.

```python
def routing_function(state):
    if 조건:
        return "node_a"
    else:
        return "node_b"

graph.add_conditional_edges("source_node", routing_function)
```

### State 채널 유형| 유형 | 정의 방식 | 동작 | 예시 ||------|----------|------|------|| **Reducer** | `Annotated[T, func]` | 함수가 이전+새 값을 결합 | `messages` (add_messages) || **Overwrite** | `T` (일반 타입) | 새 값이 이전 값을 덮어씀 | `turn_count: int` || **Default** | `T = default_value` | 초기값 지정 가능 | `mode: str = "normal"` |> `messages` 이외에 커스텀 reducer를 정의할 수도 있습니다.> 예를 들어, 숫자를 누적하는 `operator.add` reducer를 사용할 수 있습니다.>> ```python> from operator import add> tool_call_count: Annotated[int, add]  # 반환값이 기존 값에 더해짐> ```

In [ ]:
# 조건부 엣지 예시: 메시지 길이에 따라 분기
def short_reply(state: MessagesState):
    return {"messages": [AIMessage(content="짧은 질문이네요! 간단히 답할게요.")]}

def long_reply(state: MessagesState):
    return {"messages": [AIMessage(content="긴 질문이네요! 자세히 분석해볼게요.")]}

def route_by_length(state: MessagesState):
    """마지막 메시지 길이에 따라 분기합니다."""
    last_msg = state["messages"][-1].content
    if len(last_msg) > 20:
        return "long"
    return "short"

builder = StateGraph(MessagesState)
builder.add_node("short", short_reply)
builder.add_node("long", long_reply)

# START에서 조건부 분기
builder.add_conditional_edges(START, route_by_length)
builder.add_edge("short", END)
builder.add_edge("long", END)

graph = builder.compile()

# 짧은 메시지
r1 = graph.invoke({"messages": [HumanMessage(content="안녕")]})
print(f"짧은 입력: {r1['messages'][-1].content}")

# 긴 메시지
r2 = graph.invoke({"messages": [HumanMessage(content="파이썬에서 데코레이터의 동작 원리를 상세하게 설명해주세요")]})
print(f"긴 입력:   {r2['messages'][-1].content}")

## 1.5 ReAct 에이전트 구현

이제 핵심입니다. llm_node와 tool_node를 조건부 엣지로 연결하여
ReAct 패턴을 구현합니다.

```
START → llm_node → [tool_calls?] → Yes → tool_node → llm_node → ...
                                  → No  → END
```

In [ ]:
# 도구 정의
@tool
def get_weather(city: str) -> str:
    """지정된 도시의 현재 날씨 정보를 조회합니다."""
    weather_data = {
        "서울": {"temp": 15, "condition": "맑음", "humidity": 45},
        "부산": {"temp": 18, "condition": "흐림", "humidity": 65},
        "제주": {"temp": 20, "condition": "비", "humidity": 80},
    }
    data = weather_data.get(city, {"error": f"{city} 정보 없음"})
    return json.dumps(data, ensure_ascii=False)

@tool
def calculator(expression: str) -> str:
    """수학 계산을 수행합니다. 사칙연산, 거듭제곱 등을 지원합니다."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"계산 오류: {e}"

tools = [get_weather, calculator]
llm_with_tools = llm.bind_tools(tools)

print(f"도구 {len(tools)}개 바인딩 완료")

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

# ReAct 그래프 구성
def llm_node(state: MessagesState):
    """LLM을 호출합니다. 도구가 필요하면 tool_calls를 포함한 AIMessage를 반환합니다."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 그래프 빌드
builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)  # tool_calls → "tools", 없으면 → END
builder.add_edge("tools", "llm")  # 도구 실행 후 다시 LLM으로

react_graph = builder.compile()

print("ReAct 그래프 구성 완료")
print(f"노드: {list(react_graph.nodes.keys())}")

In [ ]:
# tools_condition의 내부 동작 확인# tools_condition은 마지막 AIMessage의 tool_calls를 검사합니다from langgraph.prebuilt import tools_condition# tool_calls가 있는 경우state_with_tools = {    "messages": [        HumanMessage(content="서울 날씨"),        AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"city": "서울"}, "id": "1"}]),    ]}print(f"tool_calls 있음 → '{tools_condition(state_with_tools)}'")# tool_calls가 없는 경우state_no_tools = {    "messages": [        HumanMessage(content="안녕"),        AIMessage(content="안녕하세요!"),    ]}print(f"tool_calls 없음 → '{tools_condition(state_no_tools)}'")print()print("'tools' → tool_node로 분기")print("'__end__' → END로 분기 (최종 답변)")

In [ ]:
# ReAct 에이전트 실행 — 단일 도구 호출
result = react_graph.invoke(
    {"messages": [HumanMessage(content="서울 날씨 알려줘")]}
)

print("=== 메시지 흐름 ===")
for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"  [{role}] tool_calls: {[tc['name'] for tc in msg.tool_calls]}")
    elif role == "ToolMessage":
        print(f"  [{role}] {msg.content[:80]}")
    else:
        content = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(f"  [{role}] {content[:80]}")

In [ ]:
# ReAct 에이전트 실행 — 다단계 추론
# 모델이 여러 도구를 순차적으로 호출해야 하는 질문
result = react_graph.invoke(
    {"messages": [HumanMessage(
        content="서울과 부산 날씨를 비교하고, 두 도시 기온 차이를 계산해줘"
    )]}
)

print("=== 다단계 ReAct 흐름 ===")
tool_call_count = 0
for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            tool_call_count += 1
            print(f"  [{role}] {tc['name']}({tc['args']})")
    elif role == "ToolMessage":
        print(f"  [{role}] {msg.content}")
    else:
        content = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(f"  [{role}] {content[:100]}")

print(f"\n총 도구 호출: {tool_call_count}회")

> **tools_condition의 동작**
> - 마지막 AIMessage에 `tool_calls`가 있으면 → `"tools"` 노드로 분기
> - `tool_calls`가 없으면 → `"__end__"` (END)로 분기
> - 이 조건부 엣지가 ReAct 루프의 핵심입니다

## 1.6 도구 없이 직접 답변하는 경우

도구가 바인딩되어 있어도, 모델이 도구 없이 답변할 수 있다고 판단하면
tool_calls 없이 직접 텍스트를 반환합니다.

In [ ]:
# 도구가 불필요한 질문 — 바로 END로 분기
result = react_graph.invoke(
    {"messages": [HumanMessage(content="파이썬의 리스트 컴프리헨션을 설명해줘")]}
)

print("=== 도구 불필요 — 직접 답변 ===")
msg_types = [type(m).__name__ for m in result["messages"]]
print(f"메시지 흐름: {' → '.join(msg_types)}")
print(f"ToolMessage 수: {msg_types.count('ToolMessage')}")
print(f"답변: {result['messages'][-1].content[:100]}")

### create_react_agent: 한 줄로 ReAct 만들기LangGraph는 위 패턴을 한 줄로 만들 수 있는 `create_react_agent()`를 제공합니다.```pythonfrom langgraph.prebuilt import create_react_agent# 위에서 만든 전체 코드를 한 줄로 대체agent = create_react_agent(llm, tools)result = agent.invoke({"messages": [HumanMessage(content="서울 날씨")]})```| 방식 | 장점 | 단점 ||------|------|------|| 수동 구성 | 완전한 제어, 커스텀 노드 추가 가능 | 코드가 김 || `create_react_agent` | 간결, 빠른 프로토타이핑 | 커스터마이징 제한적 |> 이 노트북에서는 **수동 구성**을 사용합니다.> 동작 원리를 이해한 후에 `create_react_agent()`를 사용하는 것을 권장합니다.

## 1.7 Checkpointer: 대화 상태 영속화

Checkpointer를 추가하면 대화 상태(메시지 이력, 도구 호출 결과 등)를
**thread_id** 단위로 저장하고 복원할 수 있습니다.

| Checkpointer | 저장 방식 | 적합한 경우 |
|-------------|----------|------------|
| `MemorySaver` | 인메모리 | 개발/테스트 |
| `SqliteSaver` | SQLite 파일 | 프로덕션 (영속) |

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Checkpointer 추가 — 그래프를 다시 빌드
memory = MemorySaver()

builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

react_with_memory = builder.compile(checkpointer=memory)

print("Checkpointer 추가 완료")

In [ ]:
# create_react_agent 비교 — 수동 vs 프리빌트from langgraph.prebuilt import create_react_agent# 한 줄로 ReAct 에이전트 생성quick_agent = create_react_agent(llm, tools)result = quick_agent.invoke(    {"messages": [HumanMessage(content="15 * 23 계산해줘")]})print("=== create_react_agent 결과 ===")print(f"답변: {result['messages'][-1].content}")print(f"메시지 수: {len(result['messages'])}개")print(f"노드: {list(quick_agent.nodes.keys())}")

In [ ]:
# thread_id로 대화 분리
config = {"configurable": {"thread_id": "thread-1"}}

# 첫 번째 질문
r1 = react_with_memory.invoke(
    {"messages": [HumanMessage(content="서울 날씨 알려줘")]},
    config=config,
)
print(f"1번째: {r1['messages'][-1].content[:80]}")

# 두 번째 질문 — 이전 대화를 기억
r2 = react_with_memory.invoke(
    {"messages": [HumanMessage(content="아까 말한 도시의 습도는?")]},
    config=config,
)
print(f"2번째: {r2['messages'][-1].content[:80]}")
print(f"\n총 메시지 수: {len(r2['messages'])}")

In [ ]:
# 다른 thread_id — 완전히 별도의 대화
config2 = {"configurable": {"thread_id": "thread-2"}}

r3 = react_with_memory.invoke(
    {"messages": [HumanMessage(content="아까 말한 도시의 습도는?")]},
    config=config2,
)
print(f"새 thread: {r3['messages'][-1].content[:100]}")
print("→ 이전 대화 기록이 없으므로 '아까'를 모릅니다")

> **thread_id 규칙**
> - 같은 thread_id → 같은 대화 이력 공유 (멀티턴)
> - 다른 thread_id → 완전히 독립된 대화
> - 채팅 앱에서는 채팅방 ID를 thread_id로 사용하는 것이 자연스럽습니다

## 1.8 커스텀 노드 추가

ReAct 그래프에 커스텀 노드를 추가하여 기능을 확장할 수 있습니다.
예를 들어:
- **요약 노드**: 대화가 길어지면 이전 대화를 요약
- **라우터 노드**: 질문 유형에 따라 다른 LLM을 선택
- **검증 노드**: LLM 답변을 검증하고 필요 시 재생성

In [ ]:
# 커스텀 노드 예시: 로깅 노드
# llm_node 전에 입력을 로깅하는 노드

call_log = []  # 로깅 저장소

def logging_node(state: MessagesState):
    """입력 메시지를 로깅합니다. 상태는 변경하지 않습니다."""
    last_msg = state["messages"][-1]
    log_entry = {
        "type": type(last_msg).__name__,
        "content": last_msg.content[:50] if isinstance(last_msg.content, str) else "...",
        "has_tool_calls": bool(getattr(last_msg, 'tool_calls', None)),
    }
    call_log.append(log_entry)
    return {"messages": []}  # 상태 변경 없음

# 로깅 노드가 포함된 그래프
builder = StateGraph(MessagesState)
builder.add_node("log", logging_node)
builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "log")
builder.add_edge("log", "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

graph_with_log = builder.compile()

call_log.clear()
result = graph_with_log.invoke(
    {"messages": [HumanMessage(content="제주 날씨 알려줘")]}
)

print("=== 로그 ===")
for entry in call_log:
    print(f"  {entry}")
print(f"\n답변: {result['messages'][-1].content[:80]}")

## 1.9 그래프 시각화

LangGraph는 그래프 구조를 시각화하는 기능을 제공합니다.
`get_graph().draw_mermaid()`로 Mermaid 다이어그램을 생성할 수 있습니다.

In [ ]:
# 그래프 구조를 Mermaid 다이어그램으로 출력
print(react_graph.get_graph().draw_mermaid())

## 1.8a 스트리밍과 ReActReAct 에이전트에서 스트리밍을 사용하면, 각 노드의 실행 결과를 실시간으로 확인할 수 있습니다.`graph.stream()`은 각 노드가 완료될 때마다 이벤트를 yield합니다.

In [ ]:
# Colab에서 그래프를 이미지로 시각화
from IPython.display import Image, display

try:
    img = react_graph.get_graph().draw_mermaid_png()
    display(Image(img))
except Exception as e:
    print(f"이미지 생성 실패: {e}")
    print("Mermaid 텍스트로 대체:")
    print(react_graph.get_graph().draw_mermaid())

In [ ]:
# ReAct 에이전트 스트리밍 — 노드별 이벤트 확인for event in react_graph.stream(    {"messages": [HumanMessage(content="서울 날씨와 15+27 계산해줘")]},    stream_mode="updates",  # 노드별 업데이트만 출력):    for node_name, update in event.items():        msgs = update.get("messages", [])        for msg in msgs:            role = type(msg).__name__            if hasattr(msg, 'tool_calls') and msg.tool_calls:                calls = [f"{tc['name']}({tc['args']})" for tc in msg.tool_calls]                print(f"[{node_name}] {role} → tool_calls: {calls}")            else:                content = msg.content if isinstance(msg.content, str) else str(msg.content)                print(f"[{node_name}] {role} → {content[:80]}")

## 1.10 에이전트 설계 패턴 정리

| 패턴 | 구조 | 적합한 경우 |
|------|------|------------|
| 단일 LLM | START → llm → END | 간단한 질의응답 |
| ReAct | llm ↔ tools (루프) | 도구 활용 에이전트 |
| 라우터 | router → node_a/node_b → END | 질문 유형별 분기 |
| 파이프라인 | node_a → node_b → node_c → END | 순차 처리 |
| 하이브리드 | router → ReAct / 단일 LLM | 복합 에이전트 |

> **그래프 설계 원칙**
> - 노드는 **하나의 책임**만 가지도록 설계합니다 (단일 책임 원칙)
> - 상태(State)에 필요한 정보만 포함합니다 (과도한 상태는 디버깅을 어렵게 함)
> - 조건부 엣지의 분기 조건은 명확해야 합니다
> - 무한 루프 방지: 도구 호출 횟수에 상한을 두거나, 종료 조건을 명확히 합니다

In [ ]:
# 무한 루프 방지: 최대 도구 호출 횟수 제한
MAX_TOOL_CALLS = 5

def llm_node_with_limit(state: MessagesState):
    """도구 호출 횟수를 제한하는 LLM 노드."""
    # 현재까지의 ToolMessage 수 세기
    tool_msg_count = sum(
        1 for m in state["messages"] if isinstance(m, ToolMessage)
    )

    if tool_msg_count >= MAX_TOOL_CALLS:
        # 도구 호출 제한 도달 → 도구 없이 답변 강제
        response = llm.invoke(state["messages"])
    else:
        response = llm_with_tools.invoke(state["messages"])

    return {"messages": [response]}

print(f"최대 도구 호출 횟수: {MAX_TOOL_CALLS}")
print("제한에 도달하면 도구 바인딩 없이 LLM을 호출하여 루프를 종료합니다")

## 1.11 SqliteSaver: 영속적 상태 저장

`MemorySaver`는 프로세스가 종료되면 상태가 사라집니다.
`SqliteSaver`를 사용하면 SQLite 파일에 상태를 영속적으로 저장합니다.

### 단계별 실행 디버깅`stream()`에서 `stream_mode="values"`를 사용하면 각 단계의 전체 상태를 확인할 수 있습니다.그래프가 예상대로 동작하는지 디버깅할 때 유용합니다.

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# SQLite 파일 기반 checkpointer
conn = sqlite3.connect("chat_state.db", check_same_thread=False)
sqlite_saver = SqliteSaver(conn)

# 그래프에 적용
builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

react_persistent = builder.compile(checkpointer=sqlite_saver)

# 대화
config = {"configurable": {"thread_id": "persistent-1"}}
r = react_persistent.invoke(
    {"messages": [HumanMessage(content="부산 날씨 알려줘")]},
    config=config,
)
print(f"답변: {r['messages'][-1].content[:80]}")
print(f"\n상태가 chat_state.db에 저장되었습니다")
print("프로세스를 재시작해도 대화를 이어갈 수 있습니다")

conn.close()

In [ ]:
# 단계별 상태 확인 — 디버깅용print("=== 단계별 실행 ===\n")step = 0for state_snapshot in react_graph.stream(    {"messages": [HumanMessage(content="제주 날씨 알려줘")]},    stream_mode="values",):    step += 1    msgs = state_snapshot["messages"]    last = msgs[-1]    role = type(last).__name__    content = last.content if isinstance(last.content, str) else str(last.content)    print(f"Step {step}: 메시지 {len(msgs)}개, 마지막=[{role}] {content[:60]}")print(f"\n총 {step} 단계로 완료")

---

### 생각해보기

1. ReAct 에이전트에서 모델이 도구를 반복 호출하다가 무한 루프에 빠질 수 있을까요? 이를 방지하려면 어떤 메커니즘이 필요할까요?
2. Checkpointer를 사용하면 대화 상태가 영속화되지만, 상태가 계속 커지면 어떤 문제가 생길까요? 컨텍스트 매니지먼트(노트북 7)와 어떻게 결합할 수 있을까요?
3. 커스텀 노드로 "답변 품질 검증 노드"를 추가한다면, 어디에 배치하고 어떤 조건으로 분기시키겠습니까?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 11-1: 간단한 2-노드 StateGraph

2개의 노드로 구성된 간단한 StateGraph를 만드세요.

**요구사항**
1. `greeting_node`: "안녕하세요!"를 반환
2. `farewell_node`: "좋은 하루 되세요!"를 반환
3. 흐름: START → greeting → farewell → END
4. 실행 결과에서 두 메시지가 모두 포함되어 있는지 확인

### 자주 하는 실수 모음| 실수 | 증상 | 해결 ||------|------|------|| `add_messages` 누락 | 메시지가 덮어씌워짐 | `Annotated[list, add_messages]` 사용 || tools_condition 대신 수동 분기 | tool_calls 감지 로직 직접 구현 | `tools_condition` 사용 || Checkpointer 없이 멀티턴 | 이전 대화 기억 못함 | `compile(checkpointer=...)` || 노드에서 전체 state 반환 | 불필요한 상태 덮어쓰기 | 변경된 필드만 반환 || `bind_tools()` 누락 | LLM이 도구를 모름 | `llm.bind_tools(tools)` 필수 |

In [ ]:
# TODO: 2-노드 StateGraph

# 정답
# def greeting_node(state: MessagesState):
#     return {"messages": [AIMessage(content="안녕하세요!")]}
#
# def farewell_node(state: MessagesState):
#     return {"messages": [AIMessage(content="좋은 하루 되세요!")]}
#
# builder = StateGraph(MessagesState)
# builder.add_node("greeting", greeting_node)
# builder.add_node("farewell", farewell_node)
# builder.add_edge(START, "greeting")
# builder.add_edge("greeting", "farewell")
# builder.add_edge("farewell", END)
#
# simple_graph = builder.compile()
# result = simple_graph.invoke({"messages": [HumanMessage(content="시작")]})
#
# for msg in result["messages"]:
#     print(f"[{type(msg).__name__}] {msg.content}")

print("TODO를 완성해주세요")

### 실습 11-2: 조건부 분기 구현

사용자 메시지의 키워드에 따라 다른 노드로 분기하는 그래프를 만드세요.

**요구사항**
1. `info_node`: 정보 안내 메시지 반환
2. `help_node`: 도움말 메시지 반환
3. 라우팅 함수: 메시지에 "도움" 포함 시 help, 그 외 info로 분기
4. "계정 도움이 필요해요"와 "영업 시간 알려줘"로 각각 테스트

In [ ]:
# TODO: 조건부 분기

# 정답
# def info_node(state: MessagesState):
#     return {"messages": [AIMessage(content="정보 안내: 영업 시간은 9시~18시입니다.")]}
#
# def help_node(state: MessagesState):
#     return {"messages": [AIMessage(content="도움말: 상담원 연결을 도와드리겠습니다.")]}
#
# def keyword_router(state: MessagesState):
#     last = state["messages"][-1].content
#     if "도움" in last:
#         return "help"
#     return "info"
#
# builder = StateGraph(MessagesState)
# builder.add_node("info", info_node)
# builder.add_node("help", help_node)
# builder.add_conditional_edges(START, keyword_router)
# builder.add_edge("info", END)
# builder.add_edge("help", END)
# cond_graph = builder.compile()
#
# r1 = cond_graph.invoke({"messages": [HumanMessage(content="계정 도움이 필요해요")]})
# print(f"도움 요청: {r1['messages'][-1].content}")
#
# r2 = cond_graph.invoke({"messages": [HumanMessage(content="영업 시간 알려줘")]})
# print(f"정보 요청: {r2['messages'][-1].content}")

print("TODO를 완성해주세요")

> **SqliteSaver vs AsyncSqliteSaver**> - `SqliteSaver`: 동기 실행용 (Colab, 스크립트)> - `AsyncSqliteSaver`: 비동기 실행용 (FastAPI, Streamlit 등)> - 비동기 환경에서 `SqliteSaver`를 사용하면 이벤트 루프가 블로킹될 수 있습니다>> ```python> from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver> async with AsyncSqliteSaver.from_conn_string("chat.db") as saver:>     graph = builder.compile(checkpointer=saver)> ```

### 실습 11-3: ReAct 에이전트 완성

새로운 도구를 정의하고 ReAct 에이전트를 처음부터 구성하세요.

**요구사항**
1. `search_book(title: str)` 도구: 책 정보 반환 (데모용 고정 데이터)
2. `translate_title(title: str, language: str)` 도구: 책 제목 번역 (데모용)
3. StateGraph + ToolNode + tools_condition으로 ReAct 그래프 구성
4. "해리포터 책을 검색하고 제목을 영어로 번역해줘"로 테스트

In [ ]:
# TODO: ReAct 에이전트 완성

# 정답
# @tool
# def search_book(title: str) -> str:
#     """책 제목으로 도서 정보를 검색합니다."""
#     books = {
#         "해리포터": {"title": "해리포터와 마법사의 돌", "author": "J.K. 롤링", "year": 1997},
#         "반지의 제왕": {"title": "반지의 제왕", "author": "J.R.R. 톨킨", "year": 1954},
#     }
#     for key, val in books.items():
#         if key in title:
#             return json.dumps(val, ensure_ascii=False)
#     return json.dumps({"error": "책을 찾을 수 없습니다"}, ensure_ascii=False)
#
# @tool
# def translate_title(title: str, language: str) -> str:
#     """책 제목을 지정된 언어로 번역합니다."""
#     translations = {
#         ("해리포터와 마법사의 돌", "영어"): "Harry Potter and the Philosopher's Stone",
#         ("반지의 제왕", "영어"): "The Lord of the Rings",
#     }
#     result = translations.get((title, language), f"[번역: {title} → {language}]")
#     return result
#
# book_tools = [search_book, translate_title]
# llm_books = llm.bind_tools(book_tools)
#
# def book_llm_node(state: MessagesState):
#     return {"messages": [llm_books.invoke(state["messages"])]}
#
# builder = StateGraph(MessagesState)
# builder.add_node("llm", book_llm_node)
# builder.add_node("tools", ToolNode(book_tools))
# builder.add_edge(START, "llm")
# builder.add_conditional_edges("llm", tools_condition)
# builder.add_edge("tools", "llm")
# book_graph = builder.compile()
#
# result = book_graph.invoke(
#     {"messages": [HumanMessage(content="해리포터 책을 검색하고 제목을 영어로 번역해줘")]}
# )
#
# for msg in result["messages"]:
#     role = type(msg).__name__
#     if hasattr(msg, 'tool_calls') and msg.tool_calls:
#         for tc in msg.tool_calls:
#             print(f"[{role}] {tc['name']}({tc['args']})")
#     elif role == "ToolMessage":
#         print(f"[{role}] {msg.content}")
#     else:
#         content = msg.content if isinstance(msg.content, str) else str(msg.content)
#         print(f"[{role}] {content[:100]}")

print("TODO를 완성해주세요")

### 실습 11-4: Checkpointer로 멀티턴 대화

MemorySaver를 사용하여 멀티턴 대화를 구현하세요.

**요구사항**
1. 이론에서 만든 react_graph에 MemorySaver를 추가
2. thread_id="practice-1"로 3턴 대화 진행
3. 1턴: "서울 날씨 알려줘", 2턴: "기온이 몇 도였어?", 3턴: "감사합니다"
4. 각 턴의 답변과 전체 메시지 수를 출력

In [ ]:
# TODO: Checkpointer 멀티턴 대화

# 정답
# mem = MemorySaver()
# builder = StateGraph(MessagesState)
# builder.add_node("llm", llm_node)
# builder.add_node("tools", ToolNode(tools))
# builder.add_edge(START, "llm")
# builder.add_conditional_edges("llm", tools_condition)
# builder.add_edge("tools", "llm")
# multi_graph = builder.compile(checkpointer=mem)
#
# config = {"configurable": {"thread_id": "practice-1"}}
#
# turns = [
#     "서울 날씨 알려줘",
#     "기온이 몇 도였어?",
#     "감사합니다",
# ]
#
# for i, msg in enumerate(turns, 1):
#     result = multi_graph.invoke(
#         {"messages": [HumanMessage(content=msg)]},
#         config=config,
#     )
#     print(f"턴 {i}: {msg}")
#     print(f"  답변: {result['messages'][-1].content[:80]}")
#     print(f"  총 메시지: {len(result['messages'])}개")
#     print()

print("TODO를 완성해주세요")

### 실습 11-5: 커스텀 State로 턴 카운팅

커스텀 State를 정의하여 대화 턴 수를 추적하세요.

**요구사항**
1. `AgentState(TypedDict)`: messages + turn_count(int)
2. llm_node에서 turn_count를 1씩 증가
3. MemorySaver로 상태 유지
4. 3턴 대화 후 turn_count가 3인지 확인

In [ ]:
# TODO: 커스텀 State 턴 카운팅

# 정답
# class AgentState(TypedDict):
#     messages: Annotated[list[AnyMessage], add_messages]
#     turn_count: int
#
# def counting_llm_node(state: AgentState):
#     response = llm_with_tools.invoke(state["messages"])
#     return {
#         "messages": [response],
#         "turn_count": state.get("turn_count", 0) + 1,
#     }
#
# mem = MemorySaver()
# builder = StateGraph(AgentState)
# builder.add_node("llm", counting_llm_node)
# builder.add_node("tools", ToolNode(tools))
# builder.add_edge(START, "llm")
# builder.add_conditional_edges("llm", tools_condition)
# builder.add_edge("tools", "llm")
# counting_graph = builder.compile(checkpointer=mem)
#
# config = {"configurable": {"thread_id": "count-1"}}
# for q in ["안녕", "서울 날씨", "감사합니다"]:
#     result = counting_graph.invoke(
#         {"messages": [HumanMessage(content=q)]},
#         config=config,
#     )
#     print(f"turn_count: {result['turn_count']}, 질문: {q}")

print("TODO를 완성해주세요")

### 실습 11-6: 그래프 시각화

그래프의 구조를 Mermaid 다이어그램으로 시각화하세요.

**요구사항**
1. 실습 11-3에서 만든 book_graph의 Mermaid 다이어그램 출력
2. 노드 목록과 엣지 목록을 확인
3. (선택) draw_mermaid_png()로 이미지 시각화

In [ ]:
# TODO: 그래프 시각화

# 정답
# # 실습 11-3의 book_graph를 사용합니다. 먼저 실습 11-3을 완성하세요.
# print("=== Mermaid 다이어그램 ===")
# print(book_graph.get_graph().draw_mermaid())
#
# print("\n=== 노드 목록 ===")
# print(list(book_graph.nodes.keys()))
#
# try:
#     from IPython.display import Image, display
#     img = book_graph.get_graph().draw_mermaid_png()
#     display(Image(img))
# except Exception as e:
#     print(f"이미지 생성 실패: {e}")

print("TODO를 완성해주세요")

### 실습 11-7: SqliteSaver로 영속적 대화

SqliteSaver를 사용하여 대화 상태를 파일에 저장하세요.

**요구사항**
1. SQLite 파일명: "practice_chat.db"
2. SqliteSaver로 ReAct 그래프 구성
3. thread_id="sqlite-test"로 2턴 대화
4. DB 파일 크기를 확인하여 상태가 저장되었는지 확인

In [ ]:
# TODO: SqliteSaver 영속적 대화

# 정답
# import os
# conn = sqlite3.connect("practice_chat.db", check_same_thread=False)
# saver = SqliteSaver(conn)
#
# builder = StateGraph(MessagesState)
# builder.add_node("llm", llm_node)
# builder.add_node("tools", ToolNode(tools))
# builder.add_edge(START, "llm")
# builder.add_conditional_edges("llm", tools_condition)
# builder.add_edge("tools", "llm")
# persist_graph = builder.compile(checkpointer=saver)
#
# config = {"configurable": {"thread_id": "sqlite-test"}}
# for q in ["제주 날씨 알려줘", "기온은 몇 도야?"]:
#     result = persist_graph.invoke(
#         {"messages": [HumanMessage(content=q)]},
#         config=config,
#     )
#     print(f"Q: {q}")
#     print(f"A: {result['messages'][-1].content[:80]}\n")
#
# conn.close()
#
# db_size = os.path.getsize("practice_chat.db")
# print(f"DB 파일 크기: {db_size:,} bytes")
# print("상태가 파일에 영속적으로 저장되었습니다")

print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 11-3에서 모델이 search_book과 translate_title을 어떤 순서로 호출했나요? 순서가 다르면 결과가 달라질 수 있을까요?
2. 실습 11-5에서 turn_count를 State에 넣는 것과 노드 외부 변수로 관리하는 것의 차이는 무엇일까요? Checkpointer와 함께 사용할 때 어떤 차이가 생기나요?
3. SqliteSaver와 MemorySaver 외에, Redis나 PostgreSQL 기반 checkpointer가 필요한 상황은 어떤 경우일까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 11-1: 커스텀 노드를 추가한 확장 에이전트 설계 (난이도: 상)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

다음 구조의 확장된 ReAct 에이전트를 설계하세요.

```
START → router_node → [유형]
  → "casual" → casual_node → END
  → "normal" → llm_node ↔ tool_node → END
```

**과제**
- `router_node`: 질문이 인사/일상이면 "casual", 도구가 필요하면 "normal" 반환
- `casual_node`: 간단한 인사/대화 처리 (도구 없이)
- `llm_node ↔ tool_node`: 기존 ReAct 패턴 (날씨/계산 도구)
- 커스텀 State: messages + mode(str) + turn_count(int)
- MemorySaver로 멀티턴 지원

**테스트 시나리오**
- "안녕하세요" → casual 경로
- "서울 날씨 알려줘" → normal(ReAct) 경로
- "아까 기온이 몇 도였어?" → 이전 대화 기억 확인

**힌트**
- router_node에서 LLM을 사용하여 질문을 분류할 수 있습니다
- 또는 간단한 키워드 매칭으로 분류할 수도 있습니다

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# 여기에 코드를 작성하세요

---

### 생각해보기

1. router_node에서 키워드 매칭 대신 LLM을 사용하면 정확도가 높아지지만, 추가 API 호출 비용이 발생합니다. 이 트레이드오프를 어떻게 해결할 수 있을까요?
2. casual 경로와 normal 경로에서 서로 다른 모델(예: 가벼운 모델 vs 무거운 모델)을 사용하면 어떤 이점이 있을까요?
3. 이 그래프에 "요약 노드"를 추가하여, 대화가 10턴을 넘으면 이전 대화를 요약하도록 설계한다면 어디에 배치하겠습니까?